# SISSO Data-Driven Modelling Workflow

This notebook provides a complete, automated workflow for running SISSO (Sure Independence Screening and Sparsifying Operator).

### Overview
SISSO is a method for finding physical descriptors (symbolic equations) that correlate with a target property. This workflow automates:
1. **Data Preparation**: Formatting `train.dat` correctly.
2. **Configuration**: Setting up `SISSO.in` parameters.
3. **Execution**: Running the compiled Fortran code.
4. **Analysis**: Extracting the best descriptors from the output.

### Prerequisites
**Crucial**: You must have the compiled `SISSO` executable in the same folder as this notebook.
- **Download Source**: [https://github.com/rouyang2017/SISSO](https://github.com/rouyang2017/SISSO)
- **Compile**: Use an MPI Fortran compiler (e.g., `mpiifort` or `mpif90`).
- **Filename**: Ensure the executable is named `SISSO` (or update the `EXECUTABLE` variable below).

In [ ]:
import pandas as pd
import numpy as np
import os
import subprocess

# ==========================================
# Global Configuration
# ==========================================
INPUT_FILE = "SISSO.in"
DATA_FILE = "train.dat"
OUTPUT_FILE = "SISSO.out"

# Path to your compiled binary.
# If using MPI, you might need to adjust the execution command later.
EXECUTABLE = "./SISSO" 

--- 
## Step 1: Data Preparation (`train.dat`)

According to your `SISSO.in`, you have:
* **18 Samples** (`nsample=18`)
* **6 Features** (`nsf=6`)

The code below generates a dummy dataset to match these requirements. In a real scenario, you would load your CSV or Excel file here.

In [ ]:
def generate_dummy_data():
    """
    Generates a dummy train.dat file that matches the dimensions
    specified in the user's SISSO.in file.
    """
    nsample = 18  # Number of samples (rows)
    nsf = 6       # Number of scalar features (columns)
    
    np.random.seed(42) # Fixed seed for reproducibility
    
    # 1. Create Sample Names (String identifiers)
    materials = [f'Mat_{i+1}' for i in range(nsample)]
    
    # 2. Create Target Property (The value Y you want to predict)
    # Generating random values between 1 and 100
    target_y = np.random.uniform(1, 100, nsample)
    
    # 3. Create Features (The inputs X)
    data = {'Material': materials, 'Target': target_y}
    for i in range(1, nsf + 1):
        data[f'Feat_{i}'] = np.random.uniform(0, 10, nsample)

    df = pd.DataFrame(data)
    
    # 4. Save to train.dat
    # SISSO Format: Space-separated values.
    # First row is header. First column is string label.
    with open(DATA_FILE, 'w') as f:
        # Write Header
        cols = list(df.columns)
        f.write(" ".join(cols) + "\n")
        
        # Write rows with specific formatting
        for _, row in df.iterrows():
            # Format material name as string, numbers with 6 decimal places
            line = f"{row['Material']} {row['Target']:.6f} "
            feats = " ".join([f"{row[c]:.6f}" for c in cols[2:]])
            f.write(line + feats + "\n")
            
    print(f"SUCCESS: Generated '{DATA_FILE}' with {nsample} samples and {nsf} features.")
    return df.head()

# Run the function
generate_dummy_data()

--- 
## Step 2: Configuration (`SISSO.in`)

Here we write the configuration file. 
* **`ptype=1`**: Regression task.
* **`desc_dim=2`**: We are looking for a 2D descriptor (a linear combination of 2 terms).
* **`ops`**: The mathematical operators allowed for feature construction.

> **Note:** The content below is exactly what you provided.

In [ ]:
sisso_in_content = """!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
! Texts after a exclamation mark (!) are comments
! The (R), (C) and (R&C) denotes the keyword used by regression, classification, and both, respectively.
! A complete list and more explanations on these keywords can be found in the SISSO_Guide.pdf
! The setting below is just an example, and user may need to change them for their jobs.
!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
ptype=1                !Property type 1: regression, 2:classification.
ntask=1                !(R&C) Multi-task learning (MTL) is invoked if >1.
scmt=.false.           !(R) Sign-Constrained MTL is invoked if .true.
desc_dim=2             !(R&C) Dimension of the descriptor/model.
nsample=18              !(R) Number of samples in train.dat. Set nsample=N1,N2,... for MTL.
!nsample=(n1,n2,...)   !(C) Number of samples. Set nsample=(n1,n2,...),(m1,m2,...),... for MTL.
restart=0              !(R&C) 0: starts from scratch, 1: continues the job(progress in the file CONTINUE)

!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
! Feature construction (FC) and sure independence screening (SIS)
! Implemented operators:(+)(-)(*)(/)(exp)(exp-)(^-1)(^2)(^3)(sqrt)(cbrt)(log)(|-|)(scd)(^6)(sin)(cos)
! scd: standard Cauchy distribution
!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
fstore=1               !(R&C) Features storage in memory. 1: by data (fast); 2: by S-expression (low-memory)
nsf= 6                 !(R&C) Number of scalar features provided in the file train.dat
ops='(+)(-)(*)(/)(exp)(exp-)'   !(R&C) Operators to be customized by user from the list shown above.
fcomplexity=3          !(R&C) Maximal feature complexity (# of operators in a feature), starting from 0.
funit=(1:1)(2:2)(3:3)(4:4)(5:6)       !(R&C) Feature unit: (n1:n2), features from n1 to n2 in train.dat have the same unit 
fmax_min=1e-3          !(R&C) The feature will be discarded if the max. abs. value in it is < fmax_min.
fmax_max=1e5           !(R&C) The feature will be discarded if the max. abs. value in it is > fmax_max.
nf_sis=50000           !(R&C) Number of features in each of the SIS-selected subspace. 

!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
! Descriptor identification (DI) via sparse regression (SO)
!>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
method_so= 'L0'        !(R&C) 'L0' or 'L1L0'(LASSO+L0). 'L0' is always recommended.
fit_intercept=.true.   !(R) Fit to a nonzero (.true.) or zero (.false.) intercept for the linear model.
metric= 'RMSE'         !(R) The metric for model selection in regression: RMSE or MaxAE (max absolute error)
nmodel=100             !(R&C) Number of the top-ranked models to output (see the folder 'Models')
isconvex=(1,1,...)     !(C) Each data group constrained to be convex domain, 1: YES; 0: NO
bwidth=0.001           !(C) Boundary tolerance for classification 
"""

with open(INPUT_FILE, "w") as f:
    f.write(sisso_in_content)
    
print(f"SUCCESS: Created '{INPUT_FILE}'.")

--- 
## Step 3: Run SISSO

This step calls the Fortran executable. 

**Troubleshooting:**
1. If you get `FileNotFoundError`, ensure the `SISSO` file is in this folder.
2. If you get `Permission denied`, run `!chmod +x SISSO` in a cell.
3. If you compiled with MPI, you may need to change the command to `['mpirun', '-np', '4', EXECUTABLE]`.

In [ ]:
def run_sisso_model():
    # Check if executable exists
    if not os.path.exists(EXECUTABLE):
        print(f"ERROR: The file '{EXECUTABLE}' does not exist.")
        print("Please compile the Fortran code and place the executable here.")
        return

    print("Running SISSO... (This may take time depending on dataset size)")
    
    try:
        # Execute the binary using subprocess
        # 'capture_output=True' captures the standard output and errors
        result = subprocess.run([EXECUTABLE], 
                                capture_output=True, 
                                text=True)
        
        if result.returncode == 0:
            print("SUCCESS: SISSO finished running.")
        else:
            print("FAILURE: SISSO exited with errors.")
            print("Error Log:", result.stderr)
            print("Output Log:", result.stdout)
            
    except Exception as e:
        print(f"An execution error occurred: {e}")

run_sisso_model()

--- 
## Step 4: Analyze Results (`SISSO.out`)

The output file `SISSO.out` contains all the identified models. 
Since we set `desc_dim=2`, SISSO looks for 1D models first, then 2D models.

The code below parses the file to find the **best 2D model** (lowest RMSE).

In [ ]:
def extract_results():
    if not os.path.exists(OUTPUT_FILE):
        print(f"File {OUTPUT_FILE} not found. Run Step 3 first.")
        return

    print(f"Parsing {OUTPUT_FILE} for results...\n")
    
    with open(OUTPUT_FILE, 'r') as f:
        lines = f.readlines()
        
    # SISSO output structure:
    # It lists models by dimension. We look for the keyword 'dimension: 2'
    # followed by the RMSE and the descriptor components.
    
    found_section = False
    
    for i, line in enumerate(lines):
        # Check if this line marks the start of 2D results
        if "dimension: 2" in line and "RMSE" in line:
            print("==== BEST 2D MODEL FOUND ====")
            print(line.strip())  # Prints the error metrics (RMSE, MaxAE)
            
            # The actual formula is usually on the following lines
            # We print the next 6 lines to capture coefficients and descriptors
            for j in range(1, 7):
                if i+j < len(lines):
                    content = lines[i+j].strip()
                    if content: # Only print non-empty lines
                        print(content)
            
            found_section = True
            print("==============================")
            break
            
    if not found_section:
        print("Warning: Could not automatically find the 'dimension: 2' section.")
        print("Tail of the file content:")
        print("".join(lines[-10:]))

extract_results()